In [1]:
# Import of dataset from huggingface
from datasets import load_dataset
import datasets

# Suppress info/debug logging, unnecessary output
datasets.logging.set_verbosity_error()

mnlp_dataset = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")
# Extract train and test splits
train_data = mnlp_dataset["train"]
test_data = mnlp_dataset["test"]
blind_data = mnlp_dataset["blind"]

C:\Users\ledue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Visualize dataset structure
print("Dataset structure:")
print(mnlp_dataset)

print("\nSample data:")
print(f"{mnlp_dataset['train'][0]}")

Dataset structure:
DatasetDict({
    test: Dataset({
        features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer'],
        num_rows: 2000
    })
    blind: Dataset({
        features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer'],
        num_rows: 1322
    })
    train: Dataset({
        features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer'],
        num_rows: 8000
    })
})

Sample data:
{'wikipedia_title': 'National Air and Space Museum', 'wikidata_id': 'Q752669', 'query': 'when did the smithsonian air and space museum open', 'query_id': 'I9oLuP4O3GOI', 'candidate_chunks': ["The National Air and Space Museum of the Smithsonian Institution, also called the NASM, is a museum in Washington, D.C.. It was established in 1946 as th

## Task [B1]  
Implement the two baseline semantic search systems:  
- distilbert-base-uncased
- all-MiniLM-L6-v2

In [3]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.modules import Transformer, Pooling

# Load DistilBERT using SentenceTransformer's Transformer module
transformer = Transformer("distilbert-base-uncased")
pooling = Pooling(transformer.auto_model.config.hidden_size, pooling_mode="mean")

model_bert = SentenceTransformer(modules=[transformer, pooling])

# Load MiniLM using SentenceTransformer's built-in support for pre-trained models
model_mini = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1380.19it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\ledue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ledue\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a

In [4]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

NORMALIZE_EMBEDDINGS = True

def normalize_embedding_matrix(embeddings):
    """L2-normalize a single embedding or a matrix of embeddings."""
    embeddings = np.asarray(embeddings)
    norms = np.linalg.norm(embeddings, axis=-1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return embeddings / norms

def encode_texts(model, texts, normalize_embeddings=NORMALIZE_EMBEDDINGS, **encode_kwargs):
    """Encode text and optionally L2-normalize embeddings for retrieval."""
    embeddings = model.encode(texts, convert_to_numpy=True, **encode_kwargs)
    if normalize_embeddings:
        embeddings = normalize_embedding_matrix(embeddings)
    return embeddings

# Cosine similarity search utility function
def cosine_similarity_search(query_embedding, chunk_embeddings, top_k=3):
    """Find top-k chunks most similar to query using normalized embeddings."""
    query_embedding = normalize_embedding_matrix(query_embedding)
    chunk_embeddings = normalize_embedding_matrix(chunk_embeddings)
    similarities = cosine_similarity(query_embedding.reshape(1, -1), chunk_embeddings).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return top_indices, similarities[top_indices]

Implementation of Hit@k metric as:  
$$Hit@k = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1} [r_i \leq k]$$

In [5]:

def hit_at_k(retrieved_indices, ground_truth_indices, k):
    """Hit@k: 1 if any ground truth in top-k results, else 0"""
    top_k_results = retrieved_indices[:k]
    gt_set = set(ground_truth_indices)
    has_hit = any(item in gt_set for item in top_k_results)
    return 1 if has_hit else 0

## [A2]
Report additional performance metrics beyond Hit@k

Embeddings are L2-normalized before ranking, so the same retrieval helpers can be used consistently across baselines, fine-tuned models, and later re-ranking experiments.

Implementation of MRR@k metric as:  
$$MRR@k = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{rank_i} \cdot \mathbb{1}[rank_i \leq k]$$

In [6]:
def mrr_at_k(retrieved_indices, ground_truth_indices, k):
    """MRR@k: Reciprocal rank of first k relevant items"""

    relevant_items = set(ground_truth_indices)
    
    # Consider only first k elements
    top_k_predictions = retrieved_indices[:k]
    
    for rank, item_id in enumerate(top_k_predictions, start=1):
        if item_id in relevant_items:
            # Found a relevant item at this rank, return reciprocal
            return 1.0 / rank
            
    # No relevant item found in top-k
    return 0.0

In [7]:
# Function to evaluate retrieval using both Hit@k and MRR@k metrics
def evaluate_retrieval(queries_embeddings, chunk_embeddings, ground_truths, k_values=[1, 3, 5]):
    queries_embeddings = normalize_embedding_matrix(queries_embeddings)
    chunk_embeddings = normalize_embedding_matrix(chunk_embeddings)
    results = {}
    for k in k_values:
        hits = [hit_at_k(cosine_similarity_search(q, chunk_embeddings, k)[0], ground_truths[i], k) 
                for i, q in enumerate(queries_embeddings)]
        mrrs = [mrr_at_k(cosine_similarity_search(q, chunk_embeddings, k)[0], ground_truths[i], k) 
                for i, q in enumerate(queries_embeddings)]
        results[f'Hit@{k}'] = np.mean(hits)
        results[f'MRR@{k}'] = np.mean(mrrs)
    return results

In [8]:
# Helper function for evaluating on real dataset
def evaluate_model_on_dataset(model, dataset, model_name="Model"):
    """Evaluate model on real dataset using Hit@k and MRR@k metrics"""
    print(f"Evaluating {model_name} on {len(dataset)} test samples...")
    
    hits_per_k = {k: [] for k in [1, 3, 5]}
    mrrs_per_k = {k: [] for k in [1, 3, 5]}
    
    for idx, example in enumerate(dataset):
        query = example['query']
        candidates = example['candidate_chunks']
        answer_pos = example['answer_pos']
        
        # Encode query and candidates with L2 normalization
        query_emb = encode_texts(model, query)
        candidates_emb = encode_texts(model, candidates)
        
        # Compute similarities and ranking
        similarities = cosine_similarity(query_emb.reshape(1, -1), candidates_emb).flatten()
        ranked_indices = np.argsort(similarities)[::-1]
        
        # Use hit_at_k and mrr_at_k functions for evaluation
        for k in [1, 3, 5]:
            hits_per_k[k].append(hit_at_k(ranked_indices, [answer_pos], k))
            mrrs_per_k[k].append(mrr_at_k(ranked_indices, [answer_pos], k))
        
        if (idx + 1) % max(1, len(dataset) // 5) == 0:
            print(f"  Progress: {(idx + 1) / len(dataset) * 100:.1f}%")
    
    # Calculate average metrics
    results = {}
    for k in [1, 3, 5]:
        results[f'Hit@{k}'] = np.mean(hits_per_k[k])
        results[f'MRR@{k}'] = np.mean(mrrs_per_k[k])
    
    print(f"Evaluation completed")
    return results

In [9]:
# Evaluate Baseline 1: DistilBERT
print("="*70)
print("BASELINE 1: DistilBERT")
print("="*70 + "\n")
bert_results = evaluate_model_on_dataset(model_bert, test_data, "DistilBERT")
print("DistilBERT Results:")
for metric in sorted(bert_results.keys()):
    print(f"  {metric}: {bert_results[metric]:.4f}")

BASELINE 1: DistilBERT

Evaluating DistilBERT on 2000 test samples...
  Progress: 20.0%
  Progress: 40.0%
  Progress: 60.0%
  Progress: 80.0%
  Progress: 100.0%
Evaluation completed
DistilBERT Results:
  Hit@1: 0.1765
  Hit@3: 0.4435
  Hit@5: 0.6045
  MRR@1: 0.1765
  MRR@3: 0.2909
  MRR@5: 0.3277


In [ ]:
# Evaluate Baseline 2: all-MiniLM-L6-v2
print("="*70)
print("BASELINE 2: all-MiniLM-L6-v2")
print("="*70 + "\n")
mini_results = evaluate_model_on_dataset(model_mini, test_data, "all-MiniLM-L6-v2")
print("all-MiniLM-L6-v2 Results:")
for metric in sorted(mini_results.keys()):
    print(f"  {metric}: {mini_results[metric]:.4f}")

## Task [B2 - Triplets]
Fine-tune DistilBERT using supervised triplet loss with random negatives.

Fine-tuning strategy:
1) Create triplet dataset: (query, positive_chunk, random_negative_chunk)
2) Train using triplet loss to minimize distance between query and correct answer
3) Maximize distance between query and a randomly sampled wrong chunk
4) Evaluate on test set and include it in the final comparison

In [ ]:
# Create triplet dataset for fine-tuning
import random
from torch.utils.data import DataLoader

print("="*70)
print("PREPARING TRIPLET DATASET")
print("="*70 + "\n")

class TripletDataset:
    """Dataset that generates triplets: (anchor, positive, negative)"""
    def __init__(self, dataset):
        self.triplets = []
        for example in dataset:
            query = example['query']
            candidates = example['candidate_chunks']
            answer_pos = example['answer_pos']
            positive_chunk = candidates[answer_pos]
            negative_positions = [i for i in range(len(candidates)) if i != answer_pos]
            negative_pos = random.choice(negative_positions) # Randomly select one negative chunk
            negative_chunk = candidates[negative_pos]
            self.triplets.append((query, positive_chunk, negative_chunk))
        
        print(f"Created {len(self.triplets)} triplets from {len(dataset)} samples")
    
    def __len__(self):
        return len(self.triplets)
    
    def __getitem__(self, idx):
        return self.triplets[idx]

# Prepare triplets data from train dataset
triplet_dataset = TripletDataset(train_data)
print(f"Dataset ready with {len(triplet_dataset)} triplets\n")

In [ ]:
# Train the model with Triplet Loss
from sentence_transformers import losses, InputExample

print("="*70)
print("FINE-TUNING WITH TRIPLET LOSS")
print("="*70 + "\n")

# Create a copy of the model for fine-tuning
from copy import deepcopy
finetuned_model = deepcopy(model_bert)

# Convert triplet data to InputExample format for SentenceTransformer
train_examples = []
for anchor, positive, negative in triplet_dataset.triplets:
    train_examples.append(InputExample(texts=[anchor, positive, negative], label=0))

print(f"Created {len(train_examples)} training examples")

# Setup training parameters
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.TripletLoss(model=finetuned_model, triplet_margin=0.5)

print(f"Starting fine-tuning...")
print(f"  - Loss: TripletLoss (margin=0.5)")
print(f"  - Epochs: 3")
print(f"  - Batch size: 16")
print(f"  - Total steps: {len(train_dataloader) * 3}\n")

# Fine-tune the model
warmup_steps = int(len(train_dataloader) * 3 * 0.1)
finetuned_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=warmup_steps,
    show_progress_bar=True
)

print(f"Fine-tuning completed")

In [ ]:
# Evaluate fine-tuned model on the same test data
print("="*70)
print("EVALUATING FINE-TUNED MODEL")
print("="*70 + "\n")

finetuned_evaluation = evaluate_model_on_dataset(finetuned_model, test_data, "Fine-tuned DistilBERT")
print("Fine-tuned Model Results (Hit and MRR):")
for metric in sorted(finetuned_evaluation.keys()):
    print(f"  {metric}: {finetuned_evaluation[metric]:.4f}")

## Task [B2 - InfoNCE Standard Hard Negatives]
Fine-tune DistilBERT with InfoNCE and standard/static hard negatives.

Training strategy:
1) Mine one hard negative for each training query using the baseline DistilBERT model.
2) Keep those hard negatives fixed for the whole training run.
3) Train with `MultipleNegativesRankingLoss`, an InfoNCE-style contrastive objective.
4) As in standard InfoNCE, the other positives in the same batch also act as negatives.
5) Use temperature scaling through `scale = 1 / temperature`.

In [ ]:
# Fine-tuning with InfoNCE and standard/static hard negatives
from copy import deepcopy
from sentence_transformers import losses, InputExample
from torch.utils.data import DataLoader
import numpy as np

print("="*70)
print("B2: INFONCE WITH STANDARD HARD NEGATIVES")
print("="*70 + "\n")

def mine_hard_negative_examples(dataset, mining_model):
    """Mine one hard negative per query with the provided model."""
    train_examples = []
    mining_stats = []
    
    for example in dataset:
        query = example['query']
        candidates = example['candidate_chunks']
        answer_pos = example['answer_pos']
        positive_chunk = candidates[answer_pos]
        negative_positions = [i for i in range(len(candidates)) if i != answer_pos]
        
        query_embedding = encode_texts(mining_model, query, show_progress_bar=False)
        candidate_embeddings = encode_texts(mining_model, candidates, show_progress_bar=False)
        similarities = cosine_similarity(query_embedding.reshape(1, -1), candidate_embeddings).flatten()
        
        hard_negative_pos = max(negative_positions, key=lambda pos: similarities[pos])
        hard_negative_chunk = candidates[hard_negative_pos]
        
        train_examples.append(InputExample(texts=[query, positive_chunk, hard_negative_chunk]))
        mining_stats.append({
            'positive_score': float(similarities[answer_pos]),
            'hard_negative_score': float(similarities[hard_negative_pos]),
            'hard_negative_pos': int(hard_negative_pos)
        })
    
    return train_examples, mining_stats

static_infonce_model = deepcopy(model_bert)
static_infonce_epochs = 3
static_infonce_batch_size = 16
static_infonce_temperature = 0.05
static_infonce_scale = 1 / static_infonce_temperature

static_infonce_examples, static_infonce_stats = mine_hard_negative_examples(
    train_data,
    model_bert
)

avg_positive_score = np.mean([item['positive_score'] for item in static_infonce_stats])
avg_hard_negative_score = np.mean([item['hard_negative_score'] for item in static_infonce_stats])

print(f"Created {len(static_infonce_examples)} static InfoNCE training examples")
print(f"Avg positive score:      {avg_positive_score:.4f}")
print(f"Avg hard negative score: {avg_hard_negative_score:.4f}")
print(f"Temperature:             {static_infonce_temperature}")
print(f"Loss scale:              {static_infonce_scale:.1f}")
print(f"Batch size:              {static_infonce_batch_size}\n")

static_infonce_dataloader = DataLoader(
    static_infonce_examples,
    shuffle=True,
    batch_size=static_infonce_batch_size
)

static_infonce_loss = losses.MultipleNegativesRankingLoss(
    model=static_infonce_model,
    scale=static_infonce_scale
)

warmup_steps = max(1, int(len(static_infonce_dataloader) * static_infonce_epochs * 0.1))

static_infonce_model.fit(
    train_objectives=[(static_infonce_dataloader, static_infonce_loss)],
    epochs=static_infonce_epochs,
    warmup_steps=warmup_steps,
    show_progress_bar=True
)

print("Static InfoNCE fine-tuning completed")
print("Model available as: static_infonce_model")

## Task [B2.0 - InfoNCE Dynamic Hard Negatives]
Fine-tune DistilBERT with InfoNCE, dynamic hard negatives, in-batch negatives, embedding normalization, and temperature scaling.

Training strategy:
1) At the beginning of each epoch, use the current model to score each query against its candidate chunks.
2) Select the highest-scoring wrong chunk as the hard negative.
3) Train with `MultipleNegativesRankingLoss`, using `(query, positive_chunk, hard_negative_chunk)` examples.
4) Use temperature scaling through the loss scale parameter (`scale = 1 / temperature`).
5) Re-mine hard negatives after every epoch, so the negatives adapt as the model changes.

In [ ]:
# Fine-tuning with dynamic hard negatives and in-batch negatives
from copy import deepcopy
from sentence_transformers import losses
from torch.utils.data import DataLoader
import numpy as np

print("="*70)
print("B2.0: DYNAMIC HARD NEGATIVES + IN-BATCH NEGATIVES")
print("="*70 + "\n")

if 'mine_hard_negative_examples' not in globals():
    raise NameError("Run the static InfoNCE cell first to define mine_hard_negative_examples.")

dynamic_hn_model = deepcopy(model_bert)
dynamic_hn_epochs = 3
dynamic_hn_batch_size = 16
dynamic_hn_temperature = 0.05
dynamic_hn_scale = 1 / dynamic_hn_temperature

# MultipleNegativesRankingLoss uses the other positives in the same batch as negatives.
# The third text in each InputExample adds one explicit hard negative per query.
dynamic_hn_loss = losses.MultipleNegativesRankingLoss(
    model=dynamic_hn_model,
    scale=dynamic_hn_scale
)

print(f"Temperature: {dynamic_hn_temperature}")
print(f"Loss scale:  {dynamic_hn_scale:.1f}")
print(f"Batch size:   {dynamic_hn_batch_size}\n")

for epoch in range(dynamic_hn_epochs):
    print(f"Epoch {epoch + 1}/{dynamic_hn_epochs}: mining hard negatives with current model...")
    dynamic_hn_examples, mining_stats = mine_hard_negative_examples(train_data, dynamic_hn_model)
    
    avg_positive_score = np.mean([item['positive_score'] for item in mining_stats])
    avg_hard_negative_score = np.mean([item['hard_negative_score'] for item in mining_stats])
    print(f"  Created {len(dynamic_hn_examples)} training examples")
    print(f"  Avg positive score:      {avg_positive_score:.4f}")
    print(f"  Avg hard negative score: {avg_hard_negative_score:.4f}")
    
    dynamic_hn_dataloader = DataLoader(
        dynamic_hn_examples,
        shuffle=True,
        batch_size=dynamic_hn_batch_size
    )
    warmup_steps = max(1, int(len(dynamic_hn_dataloader) * 0.1))
    
    dynamic_hn_model.fit(
        train_objectives=[(dynamic_hn_dataloader, dynamic_hn_loss)],
        epochs=1,
        warmup_steps=warmup_steps,
        show_progress_bar=True
    )
    print()

print("B2.0 fine-tuning completed")
print("Model available as: dynamic_hn_model")

## Overall Comparison
Compare the three main fine-tuning strategies on the same test split: Triplets, InfoNCE with standard/static hard negatives, and InfoNCE with dynamic hard negatives plus in-batch negatives.

This section includes re-ranking only if `reranked_dynamic_hn_results` already exists, so the bonus experiment remains optional.

In [ ]:
# Overall comparison across baselines and fine-tuned systems
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

required_variables = [
    'bert_results',
    'mini_results',
    'finetuned_evaluation',
    'static_infonce_model',
    'dynamic_hn_model'
]
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise NameError(f"Run the previous cells first. Missing variables: {missing_variables}")

print("="*70)
print("OVERALL COMPARISON")
print("="*70 + "\n")

if 'static_infonce_evaluation' not in globals():
    static_infonce_evaluation = evaluate_model_on_dataset(
        static_infonce_model,
        test_data,
        "InfoNCE Standard Hard Negatives DistilBERT"
    )
else:
    print("Using existing static_infonce_evaluation results")

if 'dynamic_hn_evaluation' not in globals():
    dynamic_hn_evaluation = evaluate_model_on_dataset(
        dynamic_hn_model,
        test_data,
        "Dynamic Hard Negatives DistilBERT"
    )
else:
    print("Using existing dynamic_hn_evaluation results")

overall_results = {
    'DistilBERT baseline': bert_results,
    'MiniLM baseline': mini_results,
    'Triplets': finetuned_evaluation,
    'InfoNCE standard hard negatives': static_infonce_evaluation,
    'InfoNCE dynamic hard negatives + in-batch': dynamic_hn_evaluation
}

if 'reranked_dynamic_hn_results' in globals():
    overall_results['Bonus B2.0 + cross-encoder re-ranking'] = reranked_dynamic_hn_results
else:
    print("Re-ranking results not found: bonus model is not included in this table.\n")

comparison_metrics = ['Hit@1', 'Hit@3', 'Hit@5', 'MRR@1', 'MRR@3', 'MRR@5']
comparison_table = pd.DataFrame(overall_results).T[comparison_metrics]

print("Raw scores:")
display(comparison_table.round(4))

baseline_scores = comparison_table.loc['DistilBERT baseline']
improvement_table = comparison_table.subtract(baseline_scores, axis=1)

print("Absolute improvement over DistilBERT baseline:")
display(improvement_table.round(4))

print("Best model per metric:")
for metric in comparison_metrics:
    best_model = comparison_table[metric].idxmax()
    best_score = comparison_table.loc[best_model, metric]
    print(f"  {metric}: {best_model} ({best_score:.4f})")

# Compact visualization for Hit@k and MRR@k metrics
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

hit_metrics = ['Hit@1', 'Hit@3', 'Hit@5']
mrr_metrics = ['MRR@1', 'MRR@3', 'MRR@5']

comparison_table[hit_metrics].plot(kind='bar', ax=axes[0], rot=25)
axes[0].set_title('Hit@k comparison')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend(loc='lower right')

comparison_table[mrr_metrics].plot(kind='bar', ax=axes[1], rot=25)
axes[1].set_title('MRR@k comparison')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1)
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()